# Amazon ML Challenge 2026 - Optimized Pipeline (A100 / Colab)

**Run order:** Execute cells top to bottom.  
**Quick test (50 rows):** `QUICK_RUN = True` in Cell 4 - full pipeline in ~2 min.  
**Full run:** `WHOLE_DATA = True` after verifying the pipeline works.

> Designed for Google Colab with A100 GPU and files stored on Google Drive.

In [ ]:
# ============================================================
# CELL 1 - Mount Google Drive
# This gives Colab access to your Drive files.
# ============================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)
print("Drive mounted at /content/drive")

In [ ]:
# ============================================================
# CELL 2 - Path configuration  <-- EDIT THIS FOR YOUR DRIVE LAYOUT
# ============================================================
#
# Set PIPELINE_DIR to where your Experiment_Day2/ folder lives on Drive.
# Example layouts:
#   If cloned from GitHub:  /content/Amazon_ML_Challenge/Experiment_Day2
#   If uploaded to Drive:   /content/drive/MyDrive/hackathon/Amazon_ML_Challenge/Experiment_Day2
#
# After setting this once, everything else is derived automatically.
# ============================================================

import sys
from pathlib import Path

# -----------------------------------------------------------
#  EDIT THE LINE BELOW to match YOUR Drive folder structure
# -----------------------------------------------------------
PIPELINE_DIR = Path("/content/drive/MyDrive/hackathon/Amazon_ML_Challenge/Experiment_Day2")
# -----------------------------------------------------------

if not PIPELINE_DIR.exists():
    raise FileNotFoundError(
        f"PIPELINE_DIR not found: {PIPELINE_DIR}\n"
        "Edit the PIPELINE_DIR variable above to point to your Experiment_Day2 folder."
    )

# Add the pipeline folder to Python path so 'import config' works
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))

print(f"PIPELINE_DIR : {PIPELINE_DIR}")
print(f"Files present: {[f.name for f in sorted(PIPELINE_DIR.glob('*.py'))]}")

In [ ]:
# ============================================================
# CELL 3 - Install GPU-optimized packages
# Colab sessions reset on reconnect, so this must run every session.
# Takes ~2 minutes on first run; cached after that.
# ============================================================

!pip install -q faiss-gpu polars duckdb rapidfuzz FlagEmbedding sentence-transformers
# lightgbm and torch come pre-installed on Colab A100 images
print("Packages installed.")

In [ ]:
# ============================================================
# CELL 4 - Imports, output dirs, run helper
# Safe on a fresh machine - creates all directories if missing.
# ============================================================

import os
import subprocess
import time

import config

# Create ALL output directories before anything tries to use them
config.ensure_output_dirs()
print(f"[DIRS] Output dirs ready at: {config.OUTPUT_DIR}")


def run_script(script_name: str, required: bool = True) -> bool:
    """Run a pipeline script as a subprocess and stream its output live.

    Uses PIPELINE_DIR as the working directory and passes it via PYTHONPATH
    so that 'import config' inside each script always resolves correctly
    regardless of Colab's default CWD (/content).
    """
    script_path = PIPELINE_DIR / script_name
    if not script_path.exists():
        msg = f"Script not found: {script_path}"
        if required:
            raise FileNotFoundError(msg)
        print(f"[SKIP] {msg}")
        return False

    print(f"\n{'='*60}")
    print(f"  Running: {script_name}")
    print(f"{'='*60}")
    t0 = time.perf_counter()

    # Inherit environment and add PIPELINE_DIR to PYTHONPATH
    env = os.environ.copy()
    env["PYTHONPATH"] = str(PIPELINE_DIR) + ":" + env.get("PYTHONPATH", "")

    proc = subprocess.Popen(
        [sys.executable, str(script_path)],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        cwd=str(PIPELINE_DIR),   # scripts resolve relative paths from here
        env=env,
    )
    output_lines = []
    for line in proc.stdout:
        print(line, end="", flush=True)
        output_lines.append(line)
    proc.wait()

    elapsed = time.perf_counter() - t0
    ok     = proc.returncode == 0
    status = "OK" if ok else f"FAILED (exit {proc.returncode})"
    print(f"\n[{status}] {script_name} finished in {elapsed/60:.1f} min")

    if not ok and required:
        print("\n" + "="*60)
        print("  LAST 30 LINES OF OUTPUT (for debugging):")
        print("="*60)
        for ln in output_lines[-30:]:
            print(ln, end="")
        raise RuntimeError(
            f"{script_name} failed (exit {proc.returncode}). "
            "See output above for the full traceback."
        )
    return ok


config.print_config()

In [ ]:
# ============================================================
# CELL 5 - GPU / environment sanity check
# ============================================================

import torch

if torch.cuda.is_available():
    gpu  = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    cap  = torch.cuda.get_device_properties(0).major
    print(f"GPU      : {gpu}")
    print(f"VRAM     : {vram:.1f} GB")
    print(f"SM cap   : {cap}.x  ({'Ampere+ -> bf16 enabled' if cap >= 8 else 'pre-Ampere -> fp16'})")
    try:
        import faiss
        res = faiss.StandardGpuResources()
        print("faiss-gpu: AVAILABLE")
        del res
    except Exception as e:
        print(f"faiss-gpu: not available ({e}) -- CPU FAISS will be used (slower)")
else:
    print("WARNING: No GPU found.")
    print("Go to: Runtime -> Change runtime type -> GPU -> A100")

In [ ]:
# ============================================================
# CELL 6 - RUN MODE SELECTOR  <-- MAIN BUTTON
# ============================================================
#
#   QUICK_RUN = True   ->  50 rows per file, ~2 min total
#                          Use this to smoke-test on a fresh Colab session.
#
#   QUICK_RUN = False  ->  DEV_ROWS rows (default 2M in config.py)
#
#   WHOLE_DATA = True  ->  All 26M rows, ~3-5 hours on A100
# ============================================================

QUICK_RUN  = True    # <-- SET TO True FOR 50-ROW SMOKE TEST
WHOLE_DATA = False   # <-- Set True for full competition run

import shutil

# Patch config at runtime - no need to edit config.py manually
config.QUICK_RUN  = QUICK_RUN
config.WHOLE_DATA = WHOLE_DATA

if QUICK_RUN:
    print(f"[MODE] QUICK RUN - {config.QUICK_ROWS} rows per source file")
    print("[INFO] Clearing all previous outputs so pipeline re-runs from scratch ...")
    # rmtree is safe even if dirs don't exist (ignore_errors=True)
    for d in [config.PARQUET_DIR, config.EMBEDDINGS_DIR]:
        shutil.rmtree(d, ignore_errors=True)
    # unlink is safe on missing files (missing_ok=True)
    for p in [
        config.CANDIDATES_TRAIN_PARQUET,
        config.CANDIDATES_TEST_PARQUET,
        config.FEATURES_TRAIN_PARQUET,
        config.FEATURES_TEST_PARQUET,
        config.LGBM_MODEL_PATH,
        config.SUBMISSION_PATH,
        config.CANDIDATE_SUBMISSION_PATH,
    ]:
        p.unlink(missing_ok=True)
    # Re-create dirs after clearing
    config.ensure_output_dirs()
    print("[INFO] Outputs cleared and output/ dirs re-created.")

elif WHOLE_DATA:
    print("[MODE] FULL DATA RUN")
    print("[NOTE] NOT clearing outputs. Steps will skip if files already exist.")
    config.ensure_output_dirs()

else:
    print(f"[MODE] DEV MODE ({config.DEV_ROWS:,} rows per file)")
    config.ensure_output_dirs()

config.print_config()

In [ ]:
# ============================================================
# CELL 7 - Step 0: Convert TSV -> Parquet (zstd compressed)
# ============================================================

import polars as pl

parquet_files = [
    config.TRAIN_SOURCE1_PARQUET, config.TRAIN_SOURCE2_PARQUET,
    config.TRAIN_SOURCE3_PARQUET, config.TRAIN_GT_PARQUET,
    config.TEST_SOURCE1_PARQUET,  config.TEST_SOURCE2_PARQUET,
    config.TEST_SOURCE3_PARQUET,
]

# Check that source TSVs are accessible before starting
tsv_files = [
    config.TRAIN_SOURCE1_TSV, config.TRAIN_SOURCE2_TSV,
    config.TRAIN_SOURCE3_TSV, config.TRAIN_GT_TSV,
    config.TEST_SOURCE1_TSV,  config.TEST_SOURCE2_TSV,
    config.TEST_SOURCE3_TSV,
]
missing_tsvs = [p for p in tsv_files if not p.exists()]

if missing_tsvs:
    print("[ERROR] Source TSV files not found on Drive:")
    for p in missing_tsvs:
        print(f"  {p}")
    print("\nCheck that your Drive is mounted and DATA_DIR in config.py is correct.")
    print(f"Current DATA_DIR: {config.DATA_DIR}")
elif all(p.exists() for p in parquet_files):
    print("[SKIP] All 7 Parquet files already exist.")
    print("       Delete output/parquet/ and re-run if QUICK_RUN / DEV_ROWS changed.")
else:
    missing = [p.name for p in parquet_files if not p.exists()]
    print(f"[INFO] {len(missing)} Parquet file(s) missing: {missing}")
    run_script("00_convert_to_parquet.py")

In [ ]:
# ============================================================
# CELL 8 - Step 1: Generate embeddings (bge-m3, bf16 on A100)
# ============================================================

import numpy as np

emb_files = [
    config.EMBEDDINGS_DIR / f"{s}_{t}.npy"
    for s in ("s1", "s2", "s3") for t in ("train", "test")
]
all_emb_exist = all(p.exists() for p in emb_files)

if all_emb_exist and config.TRAIN_SOURCE1_PARQUET.exists():
    s1_rows  = pl.read_parquet(config.TRAIN_SOURCE1_PARQUET).height
    emb_rows = np.load(str(config.EMBEDDINGS_DIR / "s1_train.npy"), mmap_mode="r").shape[0]
    if emb_rows != s1_rows:
        print(f"[MISMATCH] s1_train.npy: {emb_rows:,} rows vs parquet: {s1_rows:,} rows.")
        print("  Row count changed. Delete output/embeddings/ and re-run.")
    else:
        print(f"[SKIP] All embedding files exist and match parquet ({emb_rows:,} rows).")
else:
    if torch.cuda.is_available():
        free, total = torch.cuda.mem_get_info()
        print(f"[GPU MEM] Before embed: {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")
    run_script("01_embed.py")
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        free, total = torch.cuda.mem_get_info()
        print(f"[GPU MEM] After embed : {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")

In [ ]:
# ============================================================
# CELL 9 - Step 2: Blocking (GPU FAISS + TF-IDF candidates)
# ============================================================

cands_train_ok = config.CANDIDATES_TRAIN_PARQUET.exists()
cands_test_ok  = config.CANDIDATES_TEST_PARQUET.exists()

if cands_train_ok and cands_test_ok:
    n_train = pl.read_parquet(config.CANDIDATES_TRAIN_PARQUET).height
    n_test  = pl.read_parquet(config.CANDIDATES_TEST_PARQUET).height
    print(f"[SKIP] Candidate pairs exist: train={n_train:,}  test={n_test:,}")
else:
    missing = []
    if not cands_train_ok: missing.append("train")
    if not cands_test_ok:  missing.append("test")
    print(f"[INFO] Missing candidate files for: {missing}")
    run_script("02_block.py")

In [ ]:
# ============================================================
# CELL 10 - Step 3: Feature engineering
# ============================================================

feat_train_ok = config.FEATURES_TRAIN_PARQUET.exists()
feat_test_ok  = config.FEATURES_TEST_PARQUET.exists()

if feat_train_ok and feat_test_ok:
    n_train = pl.read_parquet(config.FEATURES_TRAIN_PARQUET).height
    n_test  = pl.read_parquet(config.FEATURES_TEST_PARQUET).height
    print(f"[SKIP] Feature files exist: train={n_train:,}  test={n_test:,}")
else:
    missing = []
    if not feat_train_ok: missing.append("train")
    if not feat_test_ok:  missing.append("test")
    print(f"[INFO] Missing feature files for: {missing}")
    run_script("03_feature_engineering.py")

In [ ]:
# ============================================================
# CELL 11 - Step 4: Train LightGBM (GPU-accelerated)
# ============================================================

if config.LGBM_MODEL_PATH.exists():
    print(f"[SKIP] Model already exists: {config.LGBM_MODEL_PATH.name}")
    print("       Delete it to retrain.")
else:
    print("[INFO] Model not found - training now ...")
    run_script("04_train_lgbm.py")

In [ ]:
# ============================================================
# CELL 12 - Step 5: Evaluate F_0.5 on training set
# ============================================================

run_script("06_eval_train.py")

In [ ]:
# ============================================================
# CELL 13 - Step 6: Predict on test set + generate submission
# ============================================================

run_script("05_predict_and_submit.py")

In [ ]:
# ============================================================
# CELL 14 - Step 7: Validate submission format
# ============================================================

if not config.SUBMISSION_PATH.exists():
    print(f"[ERROR] Submission file not found: {config.SUBMISSION_PATH}")
    print("        Run Cell 13 first.")
else:
    import validate_submission

    errors, warnings = validate_submission.validate(
        matching_path=str(config.SUBMISSION_PATH),
        candidate_path=str(config.CANDIDATE_SUBMISSION_PATH),
        test_dir=str(config.TEST_DIR),
        check_ids=False,
    )

    print()
    for w in warnings:
        print(f"WARNING: {w}")

    if errors:
        print(f"\nFAIL - {len(errors)} issue(s):")
        for i, e in enumerate(errors, 1):
            print(f"  {i}. {e}")
    else:
        print("\nPASS - submission format is valid. Safe to submit.")
        print(f"\nSubmission file: {config.SUBMISSION_PATH}")